In [1]:
!pip install vina
# !apt-get update -qq
# !apt-get install -y openbabel

  Using cached vina-1.2.7.tar.gz (89 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [77 lines of output]
      fatal: not a git repository (or any of the parent directories): .git
      Version found 1.2.7 (from __init__.py)
      C:\Users\Adnan Habib\AppData\Local\Temp\pip-build-env-544t3m6d\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: Apache Software License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license

In [ ]:
# Convert the receptor PDBQT to a rigid format using obabel
# This command attempts to remove flexible residues and create a rigid receptor.
# The -xr option extracts a rigid receptor.
# The -opdbqt specifies the output format as PDBQT.
# The -O specifies the output filename.
!obabel /content/protein_receptor_fixed.pdbqt -xr -opdbqt -O /content/receptor_rigid.pdbqt

print('Fixed receptor file created as /content/receptor_rigid.pdbqt')

In [ ]:
from vina import Vina

# 1. Initialize the Vina docking engine
# 'sf_name' specifies the scoring function (Vina is the standard)
v = Vina(sf_name='vina')

# 2. Load your structural files
# Note: For the receptor, use the clean version without ligand fragments
v.set_receptor('/content/receptor_rigid.pdbqt') # Changed to use the newly created rigid receptor
v.set_ligand_from_file('/content/candidate_156294005.pdbqt')

# 3. Define the Search Space (The BCL-2 Binding Pocket)
# We use the exact center coordinates we found for the 6O0K active site
v.compute_vina_maps(
    center=[-17.4, 7.3, -18.2],
    box_size=[20.0, 20.0, 20.0]
)

# 4. Run the simulation
# exhaustiveness controls how thoroughly it searches the pocket (8 is standard)
print("[*] Simulating molecular docking interactions...")
v.dock(exhaustiveness=8, n_poses=5)

# 5. Extract and print the best score
energies = v.energies()
best_score = energies[0][0]  # The first pose contains the lowest free energy

print("\n" + "="*35)
print(f"  FINAL DOCKING SCORE: {best_score:.2f} kcal/mol")
print("="*35)

# 6. Save the results (where the molecule physically landed)
v.write_poses('docking_outputs.pdbqt', n_poses=5, overwrite=True)